# Detailliertes Finanzreporting: Apple Inc. (AAPL)

---

| |                                     |
|---|-------------------------------------|
| **Unternehmen** | Apple Inc. (AAPL)                   |
| **Zeitraum** | 01. Januar 2019 – 31. Dezember 2024 |
| **Datenquelle** | Yahoo Finance (via `yfinance`)      |
| **Sprache** | Python 3 · Jupyter Notebook         |

---

## 1. Einleitung

### Analysemotivation

Apple Inc. (AAPL) wurde für diese Analyse gewählt, weil das Unternehmen mehrere zentrale Anforderungen an ein geeignetes Analyseobjekt erfüllt:

- **Krisenresilienz:** AAPL hat globale Schocks wie die COVID-19-Pandemie (2020) und den damit verbundenen Börseneinbruch von rund 30 % erfolgreich überstanden und sich anschließend stark erholt – ein idealer Stresstest für Rendite- und Risikoanalysen.
- **Datenverfügbarkeit und -qualität:** Tägliche Handelsdaten über den gesamten Zeitraum 2019–2024 sind vollständig und zuverlässig verfügbar.

## 2. Datenbeschaffung & Aufbereitung
Alle benötigten Bibliotheken werden importiert und die Rohdaten über die Yahoo Finance API geladen.

### 2.1 Bibliotheken
Die Bibliotheken `yfinance`, `pandas` und `numpy` werden für die Datenbeschaffung, -analyse und -manipulation verwendet.

In [107]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.express as px

### 2.2 Datenbeschaffung & Aufbereitung
Die Kursdaten werden über die Bibliothek `yfinance` direkt von Yahoo Finance bezogen. Der Abruf umfasst den Zeitraum vom 01. Januar 2019 bis zum 31. Dezember 2024 und enthält die täglichen OHLCV-Daten (Open, High, Low, Close, Adjusted Close, Volume). Die Spaltennamen werden zur besseren Lesbarkeit in Kleinbuchstaben umbenannt.

In [108]:
START_DATE = "2019-01-01"
END_DATE = "2024-12-31"

# Download Apple Inc. (AAPL) Daten und Aufbereitung des DataFrames
aapl = (yf.download("AAPL", start=START_DATE, end=END_DATE, auto_adjust=False)
        .reset_index()  # Datum als Spalte statt Index
        .rename(columns={
            "Date": "date",
            "Open": "open",
            "High": "high",
            "Low": "low",
            "Close": "close",
            "Adj Close": "adjusted",
            "Volume": "volume"})
        )

# MultiIndex-Spalten von yfinance auf einfache Spaltennamen reduzieren
# um die Visualisierung zu erleichtern
aapl.columns = [col[0] for col in aapl.columns]

[*********************100%***********************]  1 of 1 completed


## 3. Datenqualität & -bereinigung

Vor der eigentlichen Analyse wird der Datensatz auf mögliche Qualitätsprobleme geprüft. Konkret wird der Schlusskurs (`close`) auf drei häufige Datenfehler untersucht:

- **Fehlende Werte (`NaN`):** Tage ohne Kursangabe, z. B. durch API-Fehler oder Handelsunterbrechungen.
- **Nullwerte:** Ein Kurs von 0 ist ökonomisch nicht plausibel und deutet auf einen Datenfehler hin.
- **Negative Werte:** Aktienkurse können definitionsgemäß nicht negativ sein.

In [109]:
issues_before = {
    'missing': aapl['close'].isna().sum().item(), # Item extrahiert den skalaren Wert
    'zeros': (aapl['close'] == 0).sum().item(),
    'negatives': (aapl['close'] < 0).sum().item()
}

In [110]:
print(f"Fehlende Werte: {issues_before['missing']}")
print(f"Nullwerte: {issues_before['zeros']}")
print(f"Negative Werte: {issues_before['negatives']}")

Fehlende Werte: 0
Nullwerte: 0
Negative Werte: 0


Das Ergebnis zeigt, dass der Datensatz keine dieser Anomalien enthält – die Daten sind vollständig und konsistent und können ohne weitere Bereinigung verwendet werden.

## 4. Detaillierte Analysen

### 4.1 Kursverlauf
Das folgende Diagramm zeigt den adjustierten Schlusskurs der Apple-Aktie im Zeitraum 2020–2024. Der adjustierte Kurs berücksichtigt Dividendenzahlungen und Aktiensplits, wodurch die langfristige Kursentwicklung vergleichbar bleibt.

Deutlich erkennbar ist der starke Einbruch im März 2020 infolge der COVID-19-Pandemie, gefolgt von einer rasanten Erholung. In den darauffolgenden Jahren stieg der Kurs kontinuierlich an und erreichte Ende 2024 ein Niveau von rund 250 USD – mehr als das Dreifache des Wertes zu Beginn des Betrachtungszeitraums.

In [111]:
adjusted_line_chart = px.line(
    aapl,
    x="date",
    y="adjusted",
    title="Apple Inc. (AAPL) – Adjustierter Schlusskurs 2019–2024",
    labels={"date": "Datum", "adjusted": "Kurs (USD)"}
)

adjusted_line_chart.update_layout(
    xaxis_title="Datum",
    yaxis_title="Kurs (USD)",
    hovermode="x unified"
)

adjusted_line_chart.show()

## 4.2 Renditeanalyse

Die täglichen Renditen bilden die **Grundlage für alle weiteren Analysen** in diesem Bericht – insbesondere für die Volatilitätsanalyse, die Risikoberechnung (Value at Risk) und den Vergleich mit der Benchmark. Die einfache Rendite ist definiert als:

$$r_t = p_t / p_{t-1} - 1$$

wobei $p_t$ der adjustierte Schlusskurs am Tag $t$ und $p_{t-1}$ der Kurs am Vortag ist.

In [112]:
returns = (aapl
           .sort_values('date')
           .assign(ret=lambda x: x['adjusted'].pct_change())
           .dropna() # Entfernt die erste Zeile, da dort die Rendite nicht berechnet werden kann (kein Vortag)
           [['date', 'ret']]
           )

### Zeitreihe der täglichen Renditen

Das folgende Diagramm zeigt die täglichen Renditen im Zeitverlauf. Phasen mit hohen Ausschlägen nach oben oder unten deuten auf erhöhte Volatilität hin – besonders deutlich erkennbar ist der COVID-Schock im März 2020, der die stärksten Kursbewegungen des gesamten Betrachtungszeitraums verursachte.

In [113]:
returns_line_chart = px.line(
    returns,
    x="date",
    y="ret",
    title="Apple Inc. (AAPL) – Tägliche Renditen 2019–2024",
    labels={"date": "Datum", "ret": "Rendite"}
)

returns_line_chart.update_layout(
    xaxis_title="Datum",
    yaxis_title="Rendite",
    yaxis_tickformat=".1%",
    hovermode="x unified"
)

returns_line_chart.show()

### Verteilung der täglichen Renditen

Das Histogramm zeigt, wie häufig bestimmte Renditebereiche aufgetreten sind. Eine enge, symmetrische Verteilung um null deutet auf stabiles Marktverhalten hin. Breite Ausläufer (sogenannte *Fat Tails*) links und rechts signalisieren, dass extreme Tagesrenditen häufiger vorkommen als eine Normalverteilung erwarten würde – ein typisches Merkmal von Aktienmärkten.

In [114]:
returns_histogram = px.histogram(
    returns,
    x="ret",
    nbins=100,
    title="Apple Inc. (AAPL) – Verteilung der täglichen Renditen 2019–2024",
    labels={"ret": "Rendite"}
)

returns_histogram.update_layout(
    xaxis_title="Rendite",
    yaxis_title="Häufigkeit",
    xaxis_tickformat=".1%",
    bargap=0.05
)

returns_histogram.show()